# AI 驱动的职位搜索与申请系统

**顶点项目（Capstone）— LLM 工程课程**

多代理流水线：简历解析 → 职位发现 → 匹配打分 → 求职信生成

**模型：** Groq `llama-3.1-8b-instant`（免费层）+ 本地 `all-MiniLM-L6-v2` 嵌入  
**课程技能：** 第 1–8 周（抓取、代理、RAG、开源模型、部署、UI）

## 练习目标

把周课概念串成可演示产品：PDF 简历 → 多源职位 → 向量初筛 + LLM 打分 → 定制求职信（DOCX）→ Gradio 界面。


## 1. 环境与依赖设置

安装本笔记本所需包，并加载 API Key / 模型常量。请先准备好 `.env` 中的 `GROQ_API_KEY`（可选 `SERPAPI_KEY`）。


In [ ]:
# 安静安装本练习依赖：LLM 客户端、向量库、嵌入、UI、PDF/DOCX、校验、HTTP/HTML 解析
%pip install -q groq chromadb sentence-transformers gradio PyMuPDF python-docx pydantic python-dotenv requests beautifulsoup4


In [ ]:
# ========== 导入库 + 加载环境变量 + 初始化 Groq 客户端 ==========

# 标准库：环境、JSON、正则、字节流、休眠、哈希、URL 编码
import os, json, re, io, time, hashlib, urllib.parse
# HTML 实体解码（抓取清洗会用到）
from html import unescape
# 路径与临时文件
from pathlib import Path
# 数据类：流水线结果容器
from dataclasses import dataclass, field
# 可选类型（HTTP 失败返回 None）
from typing import Optional

# HTTP 请求
import requests
# HTML 解析（本练习主要走 JSON API，保留导入）
from bs4 import BeautifulSoup
# 从 .env 读密钥
from dotenv import load_dotenv
# 表格展示匹配结果
import pandas as pd
# PyMuPDF：抽 PDF 文本
import fitz
# python-docx：生成求职信 DOCX
from docx import Document as DocxDocument
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
# Pydantic：职位/画像/打分模型
from pydantic import BaseModel, Field, field_validator
# Chroma：向量库 + SentenceTransformer 嵌入函数
import chromadb
from chromadb.utils import embedding_functions
# Groq：托管开源 LLM
from groq import Groq
# Gradio：演示 UI
import gradio as gr

# 加载 .env
load_dotenv()

# Groq API Key（必需）
GROQ_API_KEY  = os.getenv("GROQ_API_KEY", "")
# SerpAPI（可选：Google Jobs）
SERPAPI_KEY   = os.getenv("SERPAPI_KEY", "")
# 聊天模型 id（保持原样）
LLM_MODEL     = "llama-3.1-8b-instant"
# 本地嵌入模型名（保持原样）
EMBED_MODEL   = "all-MiniLM-L6-v2"
# 抓取去重后最多保留的职位数
MAX_JOBS      = 20
# HTTP 超时秒数
REQUEST_TIMEOUT = 15

# 缺少 Key 时立刻失败，避免后面静默报错
if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not set. Add it to a .env file or set it as an environment variable.")

# 全局 Groq 客户端，供 chat / chat_json 使用
llm = Groq(api_key=GROQ_API_KEY)


## 2. 数据模型（Pydantic）

定义职位帖、工作经历、用户画像与打分结果，让代理之间传递的是结构化对象而非裸 dict。


In [ ]:
# ========== 领域模型：JobPosting / Experience / UserProfile / ScoredJob ==========

class JobPosting(BaseModel):
    # 职位标题
    title: str
    # 公司名
    company: str
    # 地点（常为 Remote）
    location: str
    # 描述摘要
    description: str
    # 原始链接（也用于去重）
    url: str
    # 薪资字符串（各源格式不一）
    salary: str = ""
    # 发布日期
    posted_date: str = ""
    # 来源站点名
    source: str = ""

    @property
    def uid(self) -> str:
        # 用 URL 的 MD5 前 8 位当稳定短 id（求职信字典的 key）
        return hashlib.md5(self.url.encode()).hexdigest()[:8]


class Experience(BaseModel):
    # 职位名
    title: str = ""
    # 公司
    company: str = ""
    # 起止时间原文
    dates: str = ""
    # 要点列表
    bullets: list[str] = Field(default_factory=list)


class UserProfile(BaseModel):
    # 候选人姓名
    name: str = "Candidate"
    email: str = ""
    phone: str = ""
    # 技能列表
    skills: list[str] = Field(default_factory=list)
    # 结构化经历
    experiences: list[Experience] = Field(default_factory=list)
    # 教育条目
    education: list[str] = Field(default_factory=list)
    # 可量化成就（写求职信时会用）
    achievements: list[str] = Field(default_factory=list)
    # PDF 抽出的原文，便于调试
    raw_text: str = ""


class ScoredJob(BaseModel):
    # 被打分的职位
    job: JobPosting
    # 0–100 匹配分
    score: int = Field(ge=0, le=100)
    # 匹配/不匹配理由
    reasons: list[str] = Field(default_factory=list)
    # 技能缺口
    skill_gaps: list[str] = Field(default_factory=list)
    # 一句话建议
    recommendation: str = ""

    @field_validator("score", mode="before")
    @classmethod
    def clamp(cls, v):
        # LLM 可能返回字符串或越界数字 → 夹到 0..100；失败则 0
        try:
            return max(0, min(100, int(v)))
        except (TypeError, ValueError):
            return 0


## 3. LLM 实用工具

封装 Groq Chat Completions：普通文本 `chat`，以及带重试的 JSON 提取 `chat_json`。


In [ ]:
# ========== Groq 聊天封装：纯文本 + 强制 JSON 抽取 ==========

def chat(system: str, user: str, temperature: float = 0.3, max_tokens: int = 1500) -> str:
    # 一次 Chat Completions：system + user
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
    )
    # 去掉首尾空白后返回助手文本
    return response.choices[0].message.content.strip()


def chat_json(system: str, user: str, retries: int = 2) -> dict:
    # 追加「只输出 JSON」约束（英文指令保留）
    system_json = system + "\n\nRespond ONLY with valid JSON. No markdown, no explanation."
    for attempt in range(retries + 1):
        try:
            # 低温更利于稳定 JSON
            raw = chat(system_json, user, temperature=0.1)
            # 从可能夹杂的文本里抠出第一个 {...}
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if not match:
                raise ValueError("No JSON object found")
            return json.loads(match.group())
        except (json.JSONDecodeError, ValueError):
            # 最后一次仍失败 → 空 dict；否则休眠再试
            if attempt == retries:
                return {}
            time.sleep(1)
    return {}


## 4. 向量存储（RAG）

用 Chroma + `all-MiniLM-L6-v2` 把简历与职位编入向量空间，先做语义初筛再交给 LLM 精打分。


In [ ]:
# ========== Chroma 集合：索引简历/职位，按相似度取 Top-N ==========

# SentenceTransformer 嵌入函数（模型名来自 EMBED_MODEL）
_embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)


def _fresh_collections():
    # 每次新建内存客户端，避免旧数据污染
    client = chromadb.Client()
    for name in ("jobs", "profile"):
        try:
            # 若同名集合已存在则删除
            client.delete_collection(name)
        except Exception:
            pass
    # 职位集合
    job_col     = client.create_collection("jobs",    embedding_function=_embed_fn)
    # 画像集合
    profile_col = client.create_collection("profile", embedding_function=_embed_fn)
    return client, job_col, profile_col


def index_profile(profile_col, profile: UserProfile) -> None:
    # 把姓名、技能、经历、成就拼成一段可嵌入文本
    text = f"{profile.name}\nSkills: {', '.join(profile.skills)}\n"
    text += "\n".join(f"{e.title} at {e.company}" for e in profile.experiences)
    text += "\n" + "\n".join(profile.achievements)
    # 固定 id，便于 upsert 覆盖
    profile_col.upsert(ids=["user_profile"], documents=[text])


def index_jobs(job_col, jobs: list[JobPosting]) -> None:
    if not jobs:
        return
    # 文档=标题+公司+描述；metadata 带回原始下标 idx
    job_col.upsert(
        ids=[f"job_{i}" for i in range(len(jobs))],
        documents=[f"{j.title} {j.company} {j.description}" for j in jobs],
        metadatas=[{"title": j.title, "company": j.company, "url": j.url, "idx": i} for i, j in enumerate(jobs)],
    )


def retrieve_top_matches(profile: UserProfile, jobs: list[JobPosting], top_n: int = 10) -> list[JobPosting]:
    if not jobs:
        return []
    # 重建集合并写入当前画像与职位
    _, job_col, profile_col = _fresh_collections()
    index_profile(profile_col, profile)
    index_jobs(job_col, jobs)

    n = min(top_n, len(jobs))
    # 用画像文档当 query，在职位库中检索
    profile_doc = profile_col.get(ids=["user_profile"])["documents"][0]
    results = job_col.query(query_texts=[profile_doc], n_results=n)

    # metadata.idx → 原始 jobs 下标集合
    matched_indices = {m["idx"] for m in results["metadatas"][0]}
    return [j for i, j in enumerate(jobs) if i in matched_indices]


## 5. 代理 1 — 简历解析器

从 PDF 抽文本，再让 LLM 按 schema 抽出结构化 `UserProfile`。


In [ ]:
# ========== 代理 1：PDF 抽文本 + LLM 结构化解析简历 ==========

def _extract_text_from_pdf(path: str) -> str:
    # 打开 PDF
    doc = fitz.open(path)
    # 逐页 get_text，用换行拼接
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    # 压缩连续空行，去掉首尾空白
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def parse_resume(file_path: str) -> UserProfile:
    # 先拿原始文本
    raw = _extract_text_from_pdf(file_path)

    # 期望 JSON schema 形状（给模型看的说明，英文保留）
    schema = {
        "name": "string",
        "email": "string",
        "phone": "string",
        "skills": ["string"],
        "experiences": [{"title": "string", "company": "string", "dates": "string", "bullets": ["string"]}],
        "education": ["string — degree, institution, year"],
        "achievements": ["string — quantified accomplishments"],
    }

    # 截断过长简历，控制 token
    result = chat_json(
        system=f"Extract structured resume data. Output schema: {json.dumps(schema)}",
        user=f"Resume text:\n{raw[:4000]}",
    )

    # 把 experiences 列表转成 Experience 对象
    experiences = [
        Experience(**e) if isinstance(e, dict) else Experience()
        for e in result.get("experiences", [])
    ]

    # 组装 UserProfile；缺字段用默认值
    profile = UserProfile(
        name=result.get("name", "Candidate"),
        email=result.get("email", ""),
        phone=result.get("phone", ""),
        skills=result.get("skills", []),
        experiences=experiences,
        education=result.get("education", []),
        achievements=result.get("achievements", []),
        raw_text=raw,
    )
    return profile


## 6. 代理 2 — 职位抓取

并行聚合多个公开职位源（Jobicy / RemoteOK / TheMuse / 可选 SerpAPI），清洗后按 URL 去重。


In [ ]:
# ========== 代理 2：多源职位抓取、清洗与去重 ==========

# 浏览器风格请求头，降低被拒概率
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
}

def _clean(text: str) -> str:
    # 空值直接返回空串
    if not text:
        return ""
    from html import unescape
    # 反复 unescape，直到不再变化（处理双重编码）
    prev = None
    while prev != text:
        prev = text
        text = unescape(text)
    return text

def _safe_get(url: str) -> Optional[requests.Response]:
    try:
        # 统一超时与请求头
        r = requests.get(url, headers=_HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r
    except requests.RequestException:
        # 网络/HTTP 错误时返回 None，由调用方当空结果
        return None


def _scrape_jobicy(query: str) -> list[JobPosting]:
    # 用查询第一个词当 tag
    tag = query.strip().split()[0].lower()
    r = _safe_get(f"https://jobicy.com/api/v2/remote-jobs?count=20&tag={urllib.parse.quote(tag)}")
    if not r:
        return []
    jobs = []
    for item in r.json().get("jobs", []):
        jobs.append(JobPosting(
            title=_clean(item.get("jobTitle", "")),
            company=_clean(item.get("companyName", "")),
            location=item.get("jobGeo", "Remote"),
            description=_clean(item.get("jobExcerpt", ""))[:1000],
            url=item.get("url", ""),
            salary=str(item.get("annualSalaryMin", "")),
            posted_date=item.get("pubDate", "")[:10],
            source="Jobicy",
        ))
    return jobs


def _scrape_remoteok(query: str) -> list[JobPosting]:
    # RemoteOK 全量 API，再按关键词过滤
    r = _safe_get("https://remoteok.com/api")
    if not r:
        return []
    q = query.lower()
    jobs = []
    for item in r.json():
        # 第一条常为元数据，无 id 则跳过
        if not isinstance(item, dict) or "id" not in item:
            continue
        text = (item.get("position", "") + " " + " ".join(item.get("tags", []))).lower()
        # 查询词任一命中标题或 tags 即保留
        if not any(word in text for word in q.split()):
            continue
        jobs.append(JobPosting(
            title=_clean(item.get("position", "")),
            company=_clean(item.get("company", "")),
            location=item.get("location", "Remote") or "Remote",
            description=_clean(item.get("description", ""))[:1000],
            url=item.get("url", f"https://remoteok.com/l/{item.get('id', '')}"),
            salary=str(item.get("salary", "")),
            posted_date=str(item.get("date", ""))[:10],
            source="RemoteOK",
        ))
        # 单源上限 15，控制总量
        if len(jobs) >= 15:
            break
    return jobs


def _scrape_themuse(query: str) -> list[JobPosting]:
    params = {"page": 1, "descending": "true", "query": query}
    r = _safe_get(f"https://www.themuse.com/api/public/jobs?{urllib.parse.urlencode(params)}")
    if not r:
        return []
    jobs = []
    for item in r.json().get("results", []):
        locations = item.get("locations", [{}])
        location = locations[0].get("name", "Remote") if locations else "Remote"
        levels = item.get("levels", [{}])
        company = item.get("company", {}).get("name", "")
        url = item.get("refs", {}).get("landing_page", "")
        jobs.append(JobPosting(
            title=_clean(item.get("name", "")),
            company=_clean(company),
            description=_clean(item.get("contents", "") or "")[:1000],
            location=location,
            url=url,
            # 用 level 名顶薪资字段（源站未必有薪资）
            salary=levels[0].get("name", "") if levels else "",
            posted_date=item.get("publication_date", "")[:10],
            source="TheMuse",
        ))
    return jobs



def _scrape_serpapi(query: str, location: str) -> list[JobPosting]:
    # 无 Key 则跳过，不报错
    if not SERPAPI_KEY:
        return []
    params = {
        "engine": "google_jobs",
        "q": query,
        "location": location,
        "api_key": SERPAPI_KEY,
        "num": MAX_JOBS,
    }
    r = _safe_get(f"https://serpapi.com/search?{urllib.parse.urlencode(params)}")
    if not r:
        return []
    jobs = []
    for item in r.json().get("jobs_results", []):
        jobs.append(JobPosting(
            title=item.get("title", ""),
            company=item.get("company_name", ""),
            location=item.get("location", ""),
            description=item.get("description", "")[:1000],
            url=item.get("share_link", ""),
            salary=str(item.get("detected_extensions", {}).get("salary", "")),
            posted_date=item.get("detected_extensions", {}).get("posted_at", ""),
            source="Google Jobs",
        ))
    return jobs


def scrape_jobs(query: str, location: str = "Remote", remote_only: bool = True) -> list[JobPosting]:
    # 四源并行概念上的聚合（同步依次调用）
    sources = {
        "Jobicy":   _scrape_jobicy(query),
        "RemoteOK": _scrape_remoteok(query),
        "TheMuse":  _scrape_themuse(query),
        "SerpAPI":  _scrape_serpapi(query, location),
    }

    # 诊断：每源条数
    diagnostics = {name: len(jobs) for name, jobs in sources.items()}
    print(f"Scraper results: {diagnostics}")

    # 展平
    all_jobs = [j for jobs in sources.values() for j in jobs]

    # 按 URL 去重，保留首次出现
    seen, unique = set(), []
    for j in all_jobs:
        if j.url and j.url not in seen:
            seen.add(j.url)
            unique.append(j)

    print(f"Total unique jobs after dedup: {len(unique)}")
    # 截断到 MAX_JOBS（remote_only 参数保留接口兼容，原逻辑未额外过滤）
    return unique[:MAX_JOBS]


## 7. 代理 3 — 职位匹配与打分

向量召回候选后，用 LLM 按 0–100 打分并给出理由、技能缺口与建议。


In [ ]:
# ========== 代理 3：LLM 打分单岗 + 批量匹配排序 ==========

def score_job(profile: UserProfile, job: JobPosting) -> ScoredJob:
    # 期望输出字段说明（英文 schema 保留）
    schema = {
        "score": "0-100 integer",
        "reasons": ["top 3 reasons this is or is not a good fit"],
        "skill_gaps": ["skills in job description the candidate lacks"],
        "recommendation": "one sentence recommendation",
    }
    result = chat_json(
        system=(
            f"You are a career advisor. Score candidate-job fit 0-100. "
            f"Output schema: {json.dumps(schema)}"
        ),
        user=(
            f"Candidate: {profile.name}\n"
            f"Skills: {', '.join(profile.skills[:15])}\n"
            f"Experience: {' | '.join(e.title + ' at ' + e.company for e in profile.experiences[:3])}\n"
            f"Achievements: {'; '.join(profile.achievements[:2])}\n\n"
            f"Job: {job.title} at {job.company} ({job.location})\n"
            f"Description: {job.description[:600]}"
        ),
    )
    # 装入 ScoredJob；缺字段用默认
    return ScoredJob(
        job=job,
        score=result.get("score", 0),
        reasons=result.get("reasons", []),
        skill_gaps=result.get("skill_gaps", []),
        recommendation=result.get("recommendation", ""),
    )


def match_and_score(profile: UserProfile, jobs: list[JobPosting], top_n: int = 5) -> list[ScoredJob]:
    # 先向量召回 2×top_n，再 LLM 精打分
    candidates = retrieve_top_matches(profile, jobs, top_n=top_n * 2)
    scored = [score_job(profile, j) for j in candidates]
    # 按分数降序，截取 top_n
    return sorted(scored, key=lambda x: x.score, reverse=True)[:top_n]


## 8. 代理 4 — 求职信生成器

用 few-shot 风格参考写出三段定制求职信，并导出为 DOCX 字节流供下载。


In [ ]:
# ========== 代理 4：生成求职信文本 + 渲染 DOCX ==========

# 开场句风格参考（英文示例保留，供 system prompt 引用）
_COVER_LETTER_EXAMPLES = """
Example 1 — opening line:
"When I reduced model inference latency by 40% at Konga, I learned that great ML engineering is as much about systems thinking as it is about algorithms — exactly the mindset your JD describes."

Example 2 — opening line:
"Building a real-time fraud detection pipeline that processed 2M transactions daily taught me more about production ML than any course could — and that experience maps directly to what you're hiring for."
""".strip()


def generate_cover_letter(profile: UserProfile, scored: ScoredJob) -> str:
    job = scored.job
    # 没有成就时用占位英文短语
    top_achievement = profile.achievements[0] if profile.achievements else "delivering impactful results"

    # 系统提示：三段、勿套话开头等（英文保留）
    system = (
        "You are an expert career coach. Write exactly one tailored 3-paragraph cover letter. "
        "Never write more than one letter. Never add a subject line, date, or second letter. "
        "Never open with 'I am writing to express my interest'. Lead with a specific achievement. "
        f"Style reference for opening lines only:\n{_COVER_LETTER_EXAMPLES}"
    )
    user = (
        f"Candidate: {profile.name}\n"
        f"Email: {profile.email}\n"
        f"Skills: {', '.join(profile.skills[:10])}\n"
        f"Top achievement: {top_achievement}\n"
        f"Recent role: {profile.experiences[0].title + ' at ' + profile.experiences[0].company if profile.experiences else 'N/A'}\n\n"
        f"Applying to: {job.title} at {job.company} ({job.location})\n"
        f"Job description excerpt: {job.description[:500]}\n"
        f"Key fit reasons: {'; '.join(scored.reasons[:2])}"
    )
    # 稍高温度增加文采；限制 token 避免超长
    return chat(system, user, temperature=0.6, max_tokens=700)


def cover_letter_to_docx(profile: UserProfile, job: JobPosting, letter_text: str) -> bytes:
    # 新建空白文档
    doc = DocxDocument()

    # 正文字体
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(11)

    # 页边距（磅）
    for section in doc.sections:
        section.top_margin    = Pt(72)
        section.bottom_margin = Pt(72)
        section.left_margin   = Pt(90)
        section.right_margin  = Pt(90)

    # 居中加粗姓名
    name_para = doc.add_paragraph(profile.name)
    name_para.runs[0].bold = True
    name_para.runs[0].font.size = Pt(14)
    name_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # 联系方式行
    contact = " | ".join(filter(None, [profile.email, profile.phone]))
    if contact:
        cp = doc.add_paragraph(contact)
        cp.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph()

    # 称呼公司招聘团队
    hiring = doc.add_paragraph(f"Hiring Team, {job.company}")
    hiring.runs[0].bold = True

    # 事由行
    role = doc.add_paragraph(f"Re: {job.title}")
    role.runs[0].italic = True

    doc.add_paragraph()

    # 正文按空行分段写入
    for para in letter_text.strip().split("\n\n"):
        if para.strip():
            doc.add_paragraph(para.strip())

    doc.add_paragraph()
    doc.add_paragraph("Sincerely,")
    closing = doc.add_paragraph(profile.name)
    closing.runs[0].bold = True

    # 写到内存 BytesIO，返回 bytes
    buf = io.BytesIO()
    doc.save(buf)
    return buf.getvalue()


## 9. 协调器（Orchestrator）

串联：解析简历 → 抓职位 → 匹配打分 → 为 Top-N 生成求职信与 DOCX。


In [ ]:
# ========== 协调器：端到端流水线 run_pipeline ==========

@dataclass
class PipelineResult:
    # 解析出的候选人画像
    profile: UserProfile
    # 抓取去重后的职位总数
    jobs_found: int
    # 打分后的 Top 列表
    scored_jobs: list[ScoredJob]
    cover_letters: dict[str, str]      # job uid -> letter text
    cover_letter_docs: dict[str, bytes] # job uid -> docx bytes


def run_pipeline(
    resume_path: str,
    job_query: str,
    location: str = "Remote",
    remote_only: bool = True,
    top_n: int = 5,
) -> PipelineResult:
    # 1) 简历 → UserProfile
    profile = parse_resume(resume_path)
    # 2) 多源抓取
    jobs = scrape_jobs(job_query, location=location, remote_only=remote_only)

    if not jobs:
        # 无职位时仍返回画像，便于 UI 提示
        return PipelineResult(profile, 0, [], {}, {})

    # 3) 向量召回 + LLM 打分
    scored = match_and_score(profile, jobs, top_n=top_n)

    # 4) 为每个打分职位写求职信并做 DOCX
    letters, docs = {}, {}
    for s in scored:
        text = generate_cover_letter(profile, s)
        letters[s.job.uid] = text
        docs[s.job.uid]    = cover_letter_to_docx(profile, s.job, text)

    return PipelineResult(
        profile=profile,
        jobs_found=len(jobs),
        scored_jobs=scored,
        cover_letters=letters,
        cover_letter_docs=docs,
    )


## 10. Gradio 用户界面

上传 PDF、填写关键词，查看匹配表与定制求职信，并下载 DOCX。


In [ ]:
# ========== Gradio UI：加载态、跑流水线、展示与下载求职信 ==========

# 会话级状态：保存最近一次 PipelineResult
_state: dict = {}


def _set_loading():
    # 点击后立刻切换按钮/面板，提示正在搜索
    return (
        gr.update(value="Searching and scoring jobs — this takes 30–60 seconds...", visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        "",
        gr.update(interactive=False, value="Searching..."),
    )


def _run(resume_file, query, location, remote_only, top_n):
    # 未上传简历：提示并恢复按钮
    if resume_file is None:
        return (
            gr.update(value="Upload a resume PDF to continue.", visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            "",
            gr.update(interactive=True, value="Find & Score Jobs"),
        )

    # 跑完整流水线；Gradio File 对象用 .name 取本地路径
    result = run_pipeline(
        resume_path=resume_file.name,
        job_query=query,
        location=location,
        remote_only=remote_only,
        top_n=int(top_n),
    )

    # 存入全局，供查看/下载回调使用
    _state["result"] = result

    if not result.scored_jobs:
        # 区分「零抓取」与「有岗但未上榜」
        jobs_msg = f"Pipeline ran. Jobs scraped: {result.jobs_found}. "
        if result.jobs_found == 0:
            jobs_msg += "All scrapers returned 0 results — check your internet connection or try a simpler query."
        else:
            jobs_msg += "Jobs were found but none scored above threshold. Try a broader query."
        return (
            gr.update(value=jobs_msg, visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            "",
            gr.update(interactive=True, value="Find & Score Jobs"),
        )

    # 构建结果表
    import pandas as pd
    df = pd.DataFrame([
        {
            "Rank":     i + 1,
            "Score":    s.score,
            "Title":    s.job.title,
            "Company":  s.job.company,
            "Location": s.job.location,
            "Salary":   s.job.salary or "—",
            "Source":   s.job.source,
            "URL":      s.job.url,
        }
        for i, s in enumerate(result.scored_jobs)
    ])

    # 下拉选项：序号. 标题 @ 公司 (Score: n)
    job_choices = [
        f"{i+1}. {s.job.title} @ {s.job.company} (Score: {s.score})"
        for i, s in enumerate(result.scored_jobs)
    ]

    profile = result.profile
    profile_md = (
        f"**Name:** {profile.name}  \n"
        f"**Email:** {profile.email}  \n"
        f"**Skills extracted:** {', '.join(profile.skills[:12])}  \n"
        f"**Jobs found:** {result.jobs_found} → Top {len(result.scored_jobs)} scored"
    )

    return (
        gr.update(value=profile_md, visible=True),
        gr.update(value=df, visible=True),
        gr.update(choices=job_choices, value=job_choices[0], visible=True),
        gr.update(visible=True),
        "",
        gr.update(interactive=True, value="Find & Score Jobs"),
    )


def _show_letter(job_choice):
    # 根据下拉选项展示匹配细节 + 求职信正文
    result = _state.get("result")
    if not result or not job_choice:
        return "", gr.update(visible=False)
    idx = int(job_choice.split(".")[0]) - 1
    scored = result.scored_jobs[idx]
    text = result.cover_letters.get(scored.job.uid, "")
    detail = (
        f"**Match Score:** {scored.score}/100  \n"
        f"**Reasons:** {' | '.join(scored.reasons)}  \n"
        f"**Skill gaps:** {', '.join(scored.skill_gaps) or 'None identified'}  \n"
        f"**Recommendation:** {scored.recommendation}  \n\n---\n\n"
        f"{text}"
    )
    return detail, gr.update(visible=True)


def _prepare_download(job_choice):
    # 把对应 DOCX bytes 写到临时目录，交给 Gradio File
    result = _state.get("result")
    if not result or not job_choice:
        return gr.update(visible=False)
    idx = int(job_choice.split(".")[0]) - 1
    scored = result.scored_jobs[idx]
    docx_bytes = result.cover_letter_docs.get(scored.job.uid, b"")
    safe_name = scored.job.company.replace(" ", "_").replace("/", "-")
    import tempfile
    out = Path(tempfile.gettempdir()) / f"cover_letter_{safe_name}.docx"
    out.write_bytes(docx_bytes)
    return gr.update(value=str(out), visible=True)


with gr.Blocks(title="Job Search & Application System", theme=gr.themes.Soft()) as app:
    gr.Markdown("# Job Search & Application System")
    gr.Markdown(
        "Upload your resume, describe your target role, and receive "
        "scored job matches with tailored cover letters."
    )

    with gr.Row():
        with gr.Column(scale=1):
            resume_input   = gr.File(label="Resume (PDF)", file_types=[".pdf"])
            query_input    = gr.Textbox(label="Job Title / Keywords", placeholder="e.g. Machine Learning Engineer")
            location_input = gr.Textbox(label="Location", value="Remote")
            remote_toggle  = gr.Checkbox(label="Remote only", value=True)
            top_n_slider   = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Results to score")
            run_btn        = gr.Button("Find & Score Jobs", variant="primary")

        with gr.Column(scale=2):
            profile_md    = gr.Markdown(visible=False)
            jobs_table    = gr.Dataframe(visible=False, interactive=False)
            job_selector  = gr.Dropdown(label="Select a job to view cover letter", visible=False)
            view_btn      = gr.Button("View Cover Letter & Details", visible=False)
            letter_md     = gr.Markdown()
            download_file = gr.File(label="Cover Letter", visible=False, interactive=False)

    # 先 loading，再真正跑流水线（链式 then）
    run_btn.click(
        fn=_set_loading,
        inputs=[],
        outputs=[profile_md, jobs_table, job_selector, view_btn, letter_md, run_btn],
        show_progress="hidden",
        queue=False,
    ).then(
        fn=_run,
        inputs=[resume_input, query_input, location_input, remote_toggle, top_n_slider],
        outputs=[profile_md, jobs_table, job_selector, view_btn, letter_md, run_btn],
        show_progress="full",
    )
    view_btn.click(
        fn=_show_letter,
        inputs=[job_selector],
        outputs=[letter_md, download_file],
        show_progress="hidden",
    )
    # 切换下拉时预写临时 DOCX
    job_selector.change(
        fn=_prepare_download,
        inputs=[job_selector],
        outputs=[download_file],
        show_progress="hidden",
    )

# 本地启动，不创建公网 share 链接
app.launch(share=False)
